# CS229 Simplified PS3 — Unsupervised Learning

参考 CS229 K-means、PCA 与 semi-supervised EM 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：** 先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

**怎么读题：** `m` 表示样本数，`d` 表示设计矩阵的列数（含截距列时也计入）。
`X` 的每一行是一条样本，`y[i]` 是对应目标；`(m,)` 是一维数组，和 `(m, 1)` 不同。
“实现”指填写函数中的 TODO；“运行检查”指执行已经给出的调用代码；“解释”写在答题框中。
函数定义格和检查格需要分别运行。检查通过只代表已检查的条件成立，不代替解释实验结果。

## Goal

对应 Spring 2026 Lecture 9–10。通过三个可见实验理解：K-means 如何压缩颜色、
PCA 如何寻找主要方向、EM 如何在隐变量存在时交替更新。

## Setup

数据和小图像都由构建脚本生成，不需要联网。

P1 使用 `image`：shape 为 `(48,48,3)`，最后一维依次是 RGB 三个颜色通道，
数值位于 `[0,1]`。P2 使用 `points`：180 个二维点，shape 为 `(180,2)`。
P3 使用该题单独生成的一维 `x_gmm`。三题的数据不同，不需要把前一题的输出作为后一题输入。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps3_unsupervised/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


In [ ]:
points = np.loadtxt(DATA_DIR / "points.csv", delimiter=",", skiprows=1)
image = np.load(DATA_DIR / "tiny_image.npy")
print("points:", points.shape, "image:", image.shape)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].scatter(points[:, 0], points[:, 1], s=12)
axes[0].set_title("Unlabelled points")
axes[1].imshow(image)
axes[1].set_title("Original tiny image")
axes[1].axis("off")
plt.show()


## Problem 1 — K-means 与图像压缩

**课堂连接：** Lecture 9。

**目标：** 原图有许多不同颜色，我们只保留 4 个代表颜色。把一个像素的 `[R,G,B]`
当作一个三维样本，在颜色空间聚类；这里不使用像素在图像中的位置。
`image.reshape(-1,3)` 将图像变成 `(2304,3)` 的 `pixels`，下方检查格已完成这一步。

**符号：** 输入 `X` 为 `(m,d)`；`k=4` 是簇的数量；`centroids` 为 `(k,d)`，
每行是一个簇的中心；`assignment[i]` 为第 `i` 个样本所属簇的编号，取值 `0` 到 `k-1`。
一个中心对应重建图像的一种颜色。

**需要实现的三个函数：**

1. `assign_clusters(X, centroids)`：计算样本到每个中心的平方欧氏距离，
   返回最近中心的整数编号数组 `(m,)`。距离相同取编号最小的中心。
2. `update_centroids(X, assignment, k, old_centroids)`：对每个簇内的样本逐列取平均，
   返回 `(k,d)` 新中心。若某簇没有样本，保留该簇的旧中心，避免空数组均值产生 NaN。
3. `fit_kmeans(X, k, rng, max_steps=50)`：用传入的 `rng` 从 `X` 中不放回抽取 `k` 行作初始中心，
   再交替分配样本和更新中心。运行固定 50 轮即可，也可在中心不再变化时提前停止。

**损失与返回值：** 本题统一记录平均平方距离

$$J=\frac1m\sum_i\lVert X_i-c_{a_i}\rVert_2^2,$$

其中 $a_i$ 为样本的簇编号，$c_{a_i}$ 为对应中心。初始化后先分配一次并记录 `J`；
每轮更新中心后重新分配，再记录 `J`。返回 `(centroids, assignment, history)`，
分别为 `(k,d)`、`(m,)` 和一维损失数组。最终 assignment 必须对应最终中心。

**检查与交付：** 下格用中心颜色替换像素，再恢复原图 shape，展示原图与 4 色图。
保留对比图和不增加的 loss 记录，在答题框解释两步为何都不会使目标变大。
这里“压缩”指减少颜色种类，不要求写压缩文件，也不保证当前 NumPy 文件体积变小。

In [ ]:
def assign_clusters(X, centroids):
    # START TODO
    raise NotImplementedError("Return nearest-centroid index for every row")
    # END TODO

def update_centroids(X, assignment, k, old_centroids):
    # START TODO
    raise NotImplementedError("Return one mean vector per cluster; handle empty clusters")
    # END TODO

def fit_kmeans(X, k, rng, max_steps=50):
    # START TODO
    raise NotImplementedError("Initialize and alternate assignment/update")
    # END TODO


In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    pixels = image.reshape(-1, 3)
    centroids, assignment, history = fit_kmeans(pixels, 4, np.random.default_rng(229))
    compressed = centroids[assignment].reshape(image.shape)
    print("loss history:", history)
    assert len(history) >= 2
    assert np.all(np.isfinite(history))
    assert compressed.shape == image.shape
    assert np.all(np.diff(history) <= 1e-10)
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(image); axes[0].set_title("original")
    axes[1].imshow(np.clip(compressed, 0, 1)); axes[1].set_title("4 colors")
    for ax in axes: ax.axis("off")
    plt.show()
else:
    print("P1 checks are off.")


> **你的答案（请双击本格后填写）**
>
> TODO：解释为什么每次 assignment/update 都不会增加 K-means objective。

## Problem 2 — PCA 降到一维再重建

**课堂连接：** Lecture 10。

**目标：** 每个点原来要用两个坐标表示；现在找到一个方向，使每个点只保留沿该方向的
一个坐标。再用这一个坐标近似恢复原来的二维点，观察丢失了多少信息。

**需要实现的函数与步骤：**

1. `fit_pca(X, n_components=1)` 接收 `(m,d)` 数据，返回 `(mean, components)`。
   `mean` 是每列的均值，shape 为 `(d,)`。用它减去每一行，得到中心化数据 `X_centered`。
2. 本题协方差统一使用 $C=X_{centered}^\top X_{centered}/m$，shape 为 `(d,d)`。
   用 `np.linalg.eigh(C)` 得到特征值与特征向量；特征向量在返回矩阵的**列**中，
   特征值默认从小到大排列。按从大到小选前 `n_components` 个对应方向。
   `components` 保持二维 shape `(d,n_components)`，不要把单列压成一维。
3. `pca_transform(X, mean, components)`：先用同一个 `mean` 中心化，再沿所选方向投影，
   返回 `(m,n_components)` 的低维坐标 `Z`。
4. `pca_inverse_transform(Z, mean, components)`：把低维坐标映射回原空间，再加回 `mean`，
   返回 `(m,d)`。利用 shape 判断这两步乘法是否需要转置。

**本题的具体 shape：** `points` 为 `(180,2)`，`mean` 为 `(2,)`，
`components` 为 `(2,1)`，`Z` 为 `(180,1)`，重建结果为 `(180,2)`。
特征向量整体换号仍是同一个主方向，不应据此判错。

**检查与交付：** 下格报告按全部坐标元素平均的重建 MSE，对比把每个点都重建为 `mean` 的基线，
并画原始点与重建点。重建点应位于通过均值的一条直线上，误差低于均值基线。
在函数旁用注释说明投影保留了哪个方向、丢失了哪个方向的信息。

In [ ]:
def fit_pca(X, n_components=1):
    # Return mean and principal directions sorted by decreasing eigenvalue.
    # START TODO
    raise NotImplementedError("Center, form covariance, and use np.linalg.eigh")
    # END TODO

def pca_transform(X, mean, components):
    # START TODO
    raise NotImplementedError("Project centered data")
    # END TODO

def pca_inverse_transform(Z, mean, components):
    # START TODO
    raise NotImplementedError("Reconstruct in original space")
    # END TODO


In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    mean, components = fit_pca(points, n_components=1)
    Z = pca_transform(points, mean, components)
    reconstructed = pca_inverse_transform(Z, mean, components)
    error = np.mean((points - reconstructed) ** 2)
    baseline = np.mean((points - mean) ** 2)
    print("components:", components.shape, "Z:", Z.shape)
    print("reconstruction MSE:", error, "baseline:", baseline)
    assert components.shape == (2, 1)
    assert Z.shape == (len(points), 1)
    assert error < baseline
    plt.figure(figsize=(6, 5))
    plt.scatter(points[:, 0], points[:, 1], s=12, alpha=0.5, label="original")
    plt.scatter(reconstructed[:, 0], reconstructed[:, 1], s=12, label="reconstructed")
    plt.xlabel("coordinate 1")
    plt.ylabel("coordinate 2")
    plt.axis("equal")
    plt.legend()
    plt.title("PCA: 2D to 1D and back")
    plt.show()
else:
    print("P2 checks are off.")


## Problem 3 — 一维两成分 GMM 的 EM（选做）

**课堂连接：** Lecture 9–10。

**背景：** 假设每个观测数来自两个高斯分布之一，但不知道它来自哪一个。
GMM（高斯混合模型）同时估计两组分布参数和每条观测属于各组的概率。
EM 交替执行“根据当前参数估计归属”（E-step）和“根据归属更新参数”（M-step）。
本题是选做，需先理解高斯密度与条件概率；不调用 sklearn。

**数据与参数：** 下格的 `x_gmm` 为 `(200,)`，训练时不提供分组标签。
`weights`、`means`、`variances` 都为 `(2,)`，第 `k` 项分别表示
混合比例 $\pi_k$、均值 $\mu_k$、**方差** $\sigma_k^2$（不是标准差）。
权重和为 1，方差必须为正。单个成分的密度为

$$f_k(x_i)=\frac{1}{\sqrt{2\pi\sigma_k^2}}
\exp\left(-\frac{(x_i-\mu_k)^2}{2\sigma_k^2}\right).$$

**E-step 接口：** `gmm_e_step(x, weights, means, variances)` 返回 `(m,2)` 的
`responsibilities`。元素 $r_{ik}$ 表示“已知观测 $x_i$ 后，它来自成分 $k$ 的概率”：

$$r_{ik}=\frac{\pi_k f_k(x_i)}{\sum_{j=0}^1\pi_j f_j(x_i)}.$$

两列分别对应两个成分；对每一行归一化，使其和为 1。它是软分配，可以是 0.3/0.7，
不像 K-means 只返回一个簇编号。

**M-step 接口：** `gmm_m_step(x, responsibilities)` 返回三个 `(2,)` 数组
`(weights, means, variances)`。令 $N_k=\sum_i r_{ik}$，用它更新：

- 混合比例为 $N_k/m$。
- 均值是用 $r_{ik}$ 加权的 `x` 平均值，分母为 $N_k$。
- 方差是用 $r_{ik}$ 加权的平方偏差平均值，偏差必须相对**本轮新均值**计算，分母仍为 $N_k$。

**数值约定：** 更新方差时设下限 `1e-8`。给定初始化下两个成分都应有有效样本；
如果 $N_k$ 为零或归一化分母为零，应明确报错并检查实现，不要给概率随意加常数后继续。
更极端数据需要在对数域计算，本题不要求实现该扩展。

**检查与交付：** 下格已提供 30 轮循环和对数似然
$\ell=\sum_i\log(\sum_k\pi_k f_k(x_i))$ 的监测代码。
完成两函数后打开检查，保留参数与似然变化输出。每行 responsibility 之和应为 1，
参数有限、权重和为 1，似然应在数值容差内不下降。在函数旁用注释解释 soft responsibility 的含义。

In [ ]:
gmm_rng = np.random.default_rng(229)
x_gmm = np.concatenate([
    gmm_rng.normal(-2.0, 0.5, size=80),
    gmm_rng.normal(2.0, 0.8, size=120),
])

def gmm_e_step(x, weights, means, variances):
    # START TODO
    raise NotImplementedError("Return responsibilities with shape (m, k)")
    # END TODO

def gmm_m_step(x, responsibilities):
    # START TODO
    raise NotImplementedError("Return updated weights, means, and variances")
    # END TODO


In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    def gmm_log_likelihood(x, weights, means, variances):
        # Stable monitoring helper; the E/M updates remain your task.
        log_density = -0.5 * (
            np.log(2 * np.pi * variances)[None, :]
            + (x[:, None] - means[None, :]) ** 2 / variances[None, :]
        )
        return np.logaddexp.reduce(np.log(weights)[None, :] + log_density, axis=1).sum()

    weights = np.array([0.5, 0.5])
    means = np.array([-1.0, 1.0])
    variances = np.array([1.0, 1.0])
    history = [gmm_log_likelihood(x_gmm, weights, means, variances)]
    for _ in range(30):
        responsibilities = gmm_e_step(x_gmm, weights, means, variances)
        assert responsibilities.shape == (len(x_gmm), 2)
        assert np.all(np.isfinite(responsibilities))
        assert np.all(responsibilities >= 0)
        assert np.allclose(responsibilities.sum(axis=1), 1.0)
        weights, means, variances = gmm_m_step(x_gmm, responsibilities)
        assert all(a.shape == (2,) for a in (weights, means, variances))
        assert all(np.all(np.isfinite(a)) for a in (weights, means, variances))
        assert np.all(weights > 0) and np.isclose(weights.sum(), 1.0)
        assert np.all(variances > 0)
        history.append(gmm_log_likelihood(x_gmm, weights, means, variances))
    assert np.all(np.diff(history) >= -1e-8)
    print("log-likelihood:", history[0], "->", history[-1])
    print("weights:", weights)
    print("means:", means)
    print("variances:", variances)
else:
    print("P3 checks are off. This problem is optional.")


## Checks

- [ ] K-means objective 不随迭代增加。
- [ ] PCA 的 components、投影和重建 shape 全部写对。
- [ ] 我能解释 K-means hard assignment 与 GMM soft responsibility 的差别。

## Next Steps

Spring 2026 的 diffusion、representation、LLM 和 RL 暂不伪装成官方 problem set。
后续只在找到可靠课堂案例或明确标注“原创 lecture lab”时再添加。